In [1]:
!pip install nltk

In [2]:
import string
import json
import codecs
import bz2 #file format of torrented reddit dataset

import sys
import pandas as pd

import matplotlib.pyplot as plt
from collections import Counter

In [3]:
#testing file path
import os
print(os.getcwd())
print(os.listdir())

/Users/forsc1/social-media-networks-a2-group49/Notebooks
['.DS_Store', 'RC_2015-01.bz2', 'DataPreProcessing.ipynb', '.ipynb_checkpoints', 'DataCollection.ipynb', 'NetworkAnalysis.py']


In [4]:
inputFile = "RC_2015-01.bz2"
outputFile = "rawComments.json"

subreddits = ["politics", "worldnews", "geopolitics", "conservative", "socialism","capitalism","libertarian"]

maxComments=5000

comments = []

lineCount = 0

with bz2.open(inputFile, "rt", encoding="utf-8") as f:
    for line in f:

        lineCount += 1

        if lineCount % 100000 == 0:
            print("Lines checked:", lineCount)

            
        try:
            comment = json.loads(line)
        except:
            continue

        subreddit = comment.get("subreddit", "")
        body = comment.get("body", "")

        if subreddit not in subreddits:
            continue

        if body in ["[deleted]", "[removed]", ""]:
            continue

        bodyLower = body.lower()

        comments.append({
         "subreddit": subreddit,
         "parent_id": comment.get("parent_id"),
         "body": body,
         "score": comment.get("score"),
         "post_id": comment.get("link_id"),
         "comment_id": comment.get("id"),
         "author": comment.get("author"),
         "created_utc": comment.get("created_utc")
        })

        if len(comments) >= maxComments:
            break

   #printing raw data
dfComments = pd.DataFrame(comments)

idToAuthor = dict(zip(dfComments["comment_id"], dfComments["author"]))

#function for getting parent author
def getParentAuthor(parent_id):

    if not isinstance(parent_id, str):
        return None

    # t1_ means parent is another comment
    if parent_id.startswith("t1_"):

        parent_comment_id = parent_id.replace("t1_", "")

        return idToAuthor.get(parent_comment_id)

    # t3_ means parent is the original post
    return None

# adding parent author column
dfComments["parent_author"] = dfComments["parent_id"].apply(getParentAuthor)

#comment edges
dfEdges = dfComments.dropna(subset=["author", "parent_author"])
dfEdges = dfEdges[dfEdges["author"] != dfEdges["parent_author"]]
dfEdges = dfEdges[["author", "parent_author", "subreddit", "created_utc"]]
dfEdges.to_csv("reddit_edges.csv", index=False)

dfComments.to_json(outputFile, orient="records")

print("Network edges:", len(dfEdges))
print("Parent authors found:", dfComments["parent_author"].notna().sum())

print("Filtered comments:", len(dfComments))
dfComments.head()


Lines checked: 100000
Lines checked: 200000
Lines checked: 300000
Lines checked: 400000
Lines checked: 500000
Network edges: 2022
Parent authors found: 2024
Filtered comments: 5000


,subreddit,parent_id,body,score,post_id,comment_id,author,created_utc,parent_author
0,worldnews,t1_cnahbvw,The vast majority of countries in the world re...,0,t3_2qxiaz,cnas90q,uncannylizard,1420070401,None
1,worldnews,t1_cnaah51,Well maybe your country should stop being such...,2,t3_2qvgji,cnas94f,mgearliosus,1420070409,None
2,politics,t1_cnaqx5p,oh this is really interesting -- I thought NYP...,1,t3_2qwwic,cnas95y,keyhole_six,1420070411,None
3,politics,t1_cnaq1ww,"Yeah, I live here, and I don't see the submiss...",-1,t3_2qx8cd,cnas99h,c00ki3mnstr,1420070417,None
4,politics,t1_cnanmfv,"They voted for the mayor, not the police offic...",1,t3_2qx8cd,cnas9a8,DrMasterBlaster,1420070419,None
